<a href="https://colab.research.google.com/github/erickvaldezsallagos/PROCESOS-ESTOCASTICOS/blob/main/serpientesyescaleras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*ERICK VALDEZ SALLAGOS*

# Resolver analíticamente y por simulación


**¿Cuál es el número promedio de tiradas necesarias para terminar el juego de serpientes y escaleras en el tablero?**

**Solucion Analitica:**

Tomemos en cuenta lo siguiente:


*   El jugador inicia antes de la casilla 1, es decir, en el estado 0
*   En cada turno se lanza un dado de 6 caras
*   Si el resultado llega o sobrepasa la casilla 20, el juego termina




Las serpientes y escaleras son:

$$3 \to 11$$
$$15 \to 19$$
$$13 \to 4$$
$$17 \to 10$$

**Variable aleatoria**

Sea:

$$E_i = \text{# esperado de tiradas para llegar al estado }20$$

comenzando desde el estado \(i\).

Claramente:
$$E_{20}=0$$

porque el juego ya terminó.


Usamos la ecuación general (ecuacion de recurrencia):
$$E_i=1+\sum_j P_{ij}E_j$$


De lo anterior:\
Obtengamos la matriz de transicion:

$$(I-Q)E=1$$
Y la solucion es:\
$$E=(I-Q)^{-1}$$

Resolviendo el sistema lineal:
$$E_0 \approx 11.77$$

**CONCLUSION**
El número promedio de tiradas necesarias para terminar el juego es:
$$\boxed{12}$$

tiradas aproximadamente.

**SOLUCION (POR SIMULACION):**

In [24]:
import numpy as np
import sympy as sp

Definicion del tablero:

In [25]:
size = 20
jumps = {
    3: 11,   #Escalera
    15: 19,  #Escalera
    13: 4,   #Serpiente
    17: 10   #Serpiente
}


Funcion analitica:

In [39]:
def solve_analytically():

    # Estados transitorios: 0 a 19
    transitory = range(size)

    n = len(transitory)

    # Matriz Q
    Q = sp.Matrix.zeros(n, n)

    # Probabilidad del dado
    p = sp.Rational(1, 6)

    # Construcción de Q
    for i in transitory:

        for die in range(1, 7):

            target = i + die

            # Si se pasa de 20, permanece igual
            if target > size:

                Q[i, i] += p

            # Si todavía no llega a 20
            elif target < size:

                # Aplicar serpiente/escalera
                final_dest = jumps.get(target, target)

                # Guardar transición
                if final_dest < size:

                    Q[i, final_dest] += p

            # Si target == 20:
            # llega al absorbente y no entra en Q

    # Matriz identidad
    I = sp.eye(n)

    # Matriz fundamental
    N = (I - Q).inv()

    # Valor esperado desde la casilla 0
    expected_value = sum(N.row(0))

    return expected_value

Simulacion Monte Carlo:

In [40]:
def simulate(n=100000):

    results = []

    for _ in range(n):

        pos = 0
        turns = 0

        while pos < size:

            turns += 1

            # Lanzar dado
            move = np.random.randint(1, 7)

            # Solo avanzar si no excede 20
            if pos + move <= size:

                # Aplicar salto
                pos = jumps.get(pos + move,
                                pos + move)

        results.append(turns)

    return np.mean(results)

**Resultados:**

In [41]:
exact_val = solve_analytically()

sim_val = simulate(100000)

print("Valor Exacto (Fracción):")
print(exact_val)

Valor Exacto (Fracción):
450485813/38834676


In [42]:
print("\nValor Exacto (Decimal):")
print(round(float(exact_val.evalf()), 6))


Valor Exacto (Decimal):
11.600092


In [43]:
print("\nValor Simulado:")
print(round(sim_val, 6))


Valor Simulado:
11.58597


**Conclusion:**

**En ambas soluciones se puede verificar que el promedio de tiradas necesarias para terminar el juego es aproximadamente el mismo**

# EJERCICIO DEL LABERINTO

**¿Cuál es la probabilidad de que el ratón, iniciando  en la casilla 0, alcance la comida?**

**SOLUCION ANALITICA:**

Sea $p_i$ la probabilidad de que el ratón llegue a la comida (estado $7$) antes de llegar al choque eléctrico (estado $8$), iniciando desde la casilla $i$

Los estados terminales son:

$p_7 = 1,
\qquad
p_8 = 0$

Como el ratón se mueve aleatoriamente con igual probabilidad entre las salidas disponibles de cada casilla, construimos las ecuaciones de primer paso.

ECUACIONES DEL SISTEMA:\

Estado 1

Desde ($1$) puede ir a ($0,3,7$):

$p_1 = \frac13(p_0+p_3+p_7)$

Como ($p_7=1$):

$p_1 = \frac13(p_0+p_3+1)$

Estado 2

Desde ($2$) puede ir a ($0,3,8$):

$p_2 = \frac13(p_0+p_3+p_8)$

Como ($p_8=0$):

$p_2 = \frac13(p_0+p_3)$

Estado 3

Desde ($3$) puede ir a ($1,2,4,5$):

$p_3 = \frac14(p_1+p_2+p_4+p_5)$

Estado 4

Desde ($4$) puede ir a ($3,6,7$):

$p_4 = \frac13(p_3+p_6+1)$
Estado 5

Desde ($5$) puede ir a ($3,6,8$):

$p_5 = \frac13(p_3+p_6)$

Estado 6

Desde ($6$) puede ir a ($4,5$):

$p_6 = \frac12(p_4+p_5)$

Sistema de ecuaciones

\begin{aligned}
p_0 &= \frac12(p_1+p_2) \\
p_1 &= \frac13(p_0+p_3+1) \\
p_2 &= \frac13(p_0+p_3) \\
p_3 &= \frac14(p_1+p_2+p_4+p_5) \\
p_4 &= \frac13(p_3+p_6+1) \\
p_5 &= \frac13(p_3+p_6) \\
p_6 &= \frac12(p_4+p_5)
\end{aligned}



Primero resolvemos ($p_6$):

$p_6=\frac12\left[\frac13(p_3+p_6+1)+\frac13(p_3+p_6)\right]$
$p_6=\frac16(2p_3+2p_6+1)$

$6p_6 = 2p_3 + 2p_6 + 1$

$4p_6 = 2p_3 + 1$

$p_6 = \frac12 p_3 + \frac14$

Ahora sustituimos en ($p_4$):

$p_4=\frac13\left(p_3 + \frac12 p_3 + \frac14 +1\right)$

$p_4=\frac13\left(\frac32 p_3 + \frac54\right)$

$p_4=\frac12 p_3 + \frac{5}{12}$

Análogamente:

$p_5=\frac13\left(\frac32 p_3 + \frac14\right)$

$p_5=\frac12 p_3 + \frac1{12}$

Ahora sustituimos en la ecuación de ($p_3$):

$p_3=\frac14\left(p_1+p_2+p_4+p_5\right)$


Simplificando:

$p_3=\frac14\left(\frac23 p_0+\frac53 p_3+\frac56\right)$


Multiplicando por 12:

$12p_3 = 2p_0 + 5p_3 + \frac52$

$7p_3 = 2p_0 + \frac52$

$p_3 = \frac27 p_0 + \frac5{14}$

Ahora usamos la ecuación de ($p_0$):

$p_0=\frac12\left[\frac13(p_0+p_3+1)+\frac13(p_0+p_3)\right]$

$p_0=\frac16(2p_0+2p_3+1)$

$6p_0 = 2p_0 + 2p_3 +1$

$4p_0 = 2p_3 +1$

$p_0 = \frac12 p_3 + \frac14$

Sustituyendo ($p_3$):

$p_0=\frac12\left(\frac27 p_0 + \frac5{14}\right)+\frac14$

$p_0=\frac17 p_0 + \frac5{28} + \frac14$

$p_0=\frac17 p_0 + \frac{12}{28}$

$p_0=\frac17 p_0 + \frac37$

$\frac67 p_0 = \frac37$

$p_0 = \frac12$

La probabilidad de que el ratón, iniciando en la casilla ($0$), alcance la comida antes del shock es:

$
\boxed{
P(\text{alcanzar la comida}) = \frac12
}
$

Equivalentemente:

$
\boxed{
P(\text{alcanzar la comida}) = 0.5 = 50\%
}$

**SOLUCION (POR SIMULACION):**

In [1]:
import random
import sympy as sp

DEFINICION DEL LABERINTO:

In [2]:
laberinto = {
    0: [1, 2],
    1: [0, 3, 7],
    2: [0, 3, 8],
    3: [1, 2, 4, 5],
    4: [3, 6, 7],
    5: [3, 6, 8],
    6: [4, 5]
}

# Estado terminal:
# 7 -> comida
# 8 -> shock

COMIDA = 7
SHOCK = 8

De lo anterior, definamos la simulacion, tomando en cuenta la solucion analitica y el diagrama

In [3]:
def simular_camino():

    estado = 0

    while True:

        # Si llega a comida
        if estado == COMIDA:
            return 1

        # Si llega al shock
        if estado == SHOCK:
            return 0

        # Movimiento aleatorio
        estado = random.choice(laberinto[estado])

Numero de simulaciones:

In [4]:
N = 100000

# Ejecutar simulaciones
exitos = 0
for _ in range(N):
    exitos += simular_camino()


Probabilidad estimada:

In [6]:
probabilidad = exitos / N

# Resultado numerico
print("Éxitos (llegar a comida):", exitos)

Éxitos (llegar a comida): 49714
Probabilidad estimada: 0.49714

Error absoluto:
0.0028599999999999737


In [7]:
print("Probabilidad estimada:", probabilidad)

Probabilidad estimada: 0.49714


**ERROR ENTRE SOLUCION ANALITICA Y SOLUCION POR SIMULACION**

In [8]:
error = abs(probabilidad - float(resultado_analitico))

print("\nError absoluto:")
print(error)


Error absoluto:
0.0028599999999999737


**CONCLUSION:**

Ambas soluciones van entorno al mismo resultado, por la tanto podemos decir que ambas soluciones son correctas y se aproximan a $\frac{1}{2}$ o $0.5$